# Ophiolite Seismic Processing Golden Path

This notebook uses the domain-first seismic SDK surface rather than a workflow-specific helper API.

- SEG-Y ingest stays in a short setup section
- the main modeling cells use `TraceBoostApp`, `SectionSelection`, and `TraceProcessingPipeline`


In [1]:
from importlib import import_module
from pathlib import Path
import json
import shutil
import sys
import tempfile

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").exists() and (candidate / "python" / "src").exists():
            return candidate
    raise RuntimeError("could not locate the Ophiolite repo root from the current working directory")

REPO_ROOT = find_repo_root(Path.cwd())
PYTHON_SRC = REPO_ROOT / "python" / "src"
NOTEBOOK_DIR = REPO_ROOT / "examples" / "golden_paths" / "seismic_processing"
for path in reversed((PYTHON_SRC, NOTEBOOK_DIR)):
    normalized = str(path)
    if normalized in sys.path:
        sys.path.remove(normalized)
    sys.path.insert(0, normalized)

for module_name in tuple(sys.modules):
    if module_name.startswith(("ophiolite_sdk", "ophiolite_automation")):
        del sys.modules[module_name]

seismic = import_module("ophiolite_sdk.seismic")
TraceBoostApp = seismic.TraceBoostApp
SectionSelection = seismic.SectionSelection
TraceProcessingPipeline = seismic.TraceProcessingPipeline

REPO_ROOT


PosixPath('/Users/sc/dev/ophiolite')

## Data selection and import

The setup section resolves a local SEG-Y fixture, preflights it, and imports it into a temporary tbvol store.

In [2]:
def default_segy_candidates() -> list[Path]:
    return [
        REPO_ROOT / "test-data" / "small.sgy",
        REPO_ROOT / "test_data" / "small.sgy",
        REPO_ROOT.parent / "TraceBoost" / "test-data" / "small.sgy",
        Path("/Users/sc/Downloads/SubsurfaceData/blocks/F3/f3_dataset.sgy"),
    ]

def detect_default_segy_path() -> Path:
    for candidate in default_segy_candidates():
        if candidate.exists():
            return candidate
    raise FileNotFoundError("could not resolve a local SEG-Y fixture for the seismic notebook")

run_root = Path(tempfile.mkdtemp(prefix="ophiolite_seismic_notebook_"))
store_path = run_root / "input.tbvol"
processed_store_path = run_root / "input_bandpass_agc.tbvol"
output_root = run_root / "outputs"
output_root.mkdir(parents=True, exist_ok=True)

app = TraceBoostApp()
segy_path = detect_default_segy_path()
preflight = app.preflight_import(segy_path)
dataset = app.import_segy(segy_path, store_path, overwrite_existing=True, preflight=preflight)

preflight_path = output_root / "preflight.json"
dataset_path = output_root / "dataset.json"
preflight_path.write_text(json.dumps(preflight.to_payload(), indent=2) + "\n")
dataset_path.write_text(json.dumps(dataset.to_payload(), indent=2) + "\n")

{
    "segy_path": str(segy_path),
    "run_root": str(run_root),
    "preflight": {
        "layout": preflight.layout,
        "suggested_action": preflight.suggested_action,
        "resolved_geometry": preflight.resolved_geometry.to_payload(),
    },
    "dataset": {
        "id": dataset.descriptor.id,
        "label": dataset.descriptor.label,
        "shape": dataset.descriptor.shape,
        "sample_interval_ms": dataset.descriptor.sample_interval_ms,
    },
}


{'segy_path': '/Users/sc/dev/TraceBoost/test-data/small.sgy',
 'run_root': '/var/folders/__/63xydj956wb3x3734qw6cgxc0000gq/T/ophiolite_seismic_notebook_f76uf8qb',
 'preflight': {'layout': 'post_stack_3d',
  'suggested_action': 'direct_dense_ingest',
  'resolved_geometry': {'inline_3d': {'start_byte': 189, 'value_type': 'i32'},
   'crossline_3d': {'start_byte': 193, 'value_type': 'i32'}}},
 'dataset': {'id': 'input.tbvol',
  'label': 'input',
  'shape': (5, 5, 50),
  'sample_interval_ms': 4.0}}

## Section and processing pipeline

This is the main domain-facing step: select a section and define the trace-local processing pipeline in seismic terms.

In [3]:
section = dataset.midpoint_section(axis="inline")
pipeline = (
    TraceProcessingPipeline.named(
        "Bandpass + RMS AGC",
        description="Trace-local seismic golden path: bandpass filter followed by RMS AGC.",
    )
    .bandpass(8.0, 12.0, 45.0, 60.0)
    .agc_rms(40.0)
)

{
    "section": section,
    "pipeline": pipeline.to_payload(),
    "operator_ids": pipeline.operator_ids(),
}


{'section': SectionSelection(axis='inline', index=2),
 'pipeline': {'schema_version': 2,
  'revision': 1,
  'steps': [{'operation': {'bandpass_filter': {'f1_hz': 8.0,
      'f2_hz': 12.0,
      'f3_hz': 45.0,
      'f4_hz': 60.0,
      'phase': 'zero',
      'window': 'cosine_taper'}},
    'checkpoint': False},
   {'operation': {'agc_rms': {'window_ms': 40.0}}, 'checkpoint': False}],
  'name': 'Bandpass + RMS AGC',
  'description': 'Trace-local seismic golden path: bandpass filter followed by RMS AGC.'},
 'operator_ids': ('bandpass_filter', 'agc_rms')}

## Preview and materialize

The main execution steps now read as `dataset.preview_processing(...)` and `dataset.run_processing(...)`.

In [4]:
raw_section = dataset.section(section)
preview = dataset.preview_processing(section, pipeline)
processed = dataset.run_processing(
    pipeline,
    output_store_path=processed_store_path,
    overwrite_existing=True,
)
processed_section = processed.section(section)

raw_section_path = output_root / "raw_inline_section.json"
preview_path = output_root / "preview_bandpass_agc_section.json"
processed_dataset_path = output_root / "processed_dataset.json"
processed_section_path = output_root / "processed_inline_section.json"
raw_section_path.write_text(json.dumps(raw_section.to_payload(), indent=2) + "\n")
preview_path.write_text(json.dumps(preview.to_payload(), indent=2) + "\n")
processed_dataset_path.write_text(json.dumps(processed.to_payload(), indent=2) + "\n")
processed_section_path.write_text(json.dumps(processed_section.to_payload(), indent=2) + "\n")

{
    "raw_section_stats": raw_section.stats(),
    "preview": {
        "processing_label": preview.processing_label,
        "preview_ready": preview.preview_ready,
        "section_stats": preview.section.stats(),
    },
    "processed_dataset_label": processed.descriptor.label,
    "processed_section_stats": processed_section.stats(),
}


{'raw_section_stats': {'trace_count': 5,
  'sample_count': 50,
  'amplitude_count': 250,
  'min_amplitude': 3.1999998092651367,
  'max_amplitude': 3.240489959716797,
  'mean_amplitude': 3.220244647979736,
  'mean_abs_amplitude': 3.220244647979736,
  'rms_amplitude': 3.2202757045804575},
 'preview': {'processing_label': 'Bandpass + RMS AGC',
  'preview_ready': True,
  'section_stats': {'trace_count': 5,
   'sample_count': 50,
   'amplitude_count': 250,
   'min_amplitude': -1.700728178024292,
   'max_amplitude': 1.700728178024292,
   'mean_amplitude': -0.0003262000548093056,
   'mean_abs_amplitude': 0.7249563160658253,
   'rms_amplitude': 0.8647992635633047}},
 'processed_dataset_label': 'input_bandpass_agc',
 'processed_section_stats': {'trace_count': 5,
  'sample_count': 50,
  'amplitude_count': 250,
  'min_amplitude': -1.700728178024292,
  'max_amplitude': 1.700728178024292,
  'mean_amplitude': -0.0003262000548093056,
  'mean_abs_amplitude': 0.7249563160658253,
  'rms_amplitude': 0.86

In [5]:
preview.to_payload()


{'schema_version': 1,
 'preview': {'section': {'dataset_id': 'input.tbvol',
   'axis': 'inline',
   'coordinate': {'index': 2, 'value': 3.0},
   'traces': 5,
   'samples': 50,
   'horizontal_axis_f64le': [0,
    0,
    0,
    0,
    0,
    0,
    52,
    64,
    0,
    0,
    0,
    0,
    0,
    0,
    53,
    64,
    0,
    0,
    0,
    0,
    0,
    0,
    54,
    64,
    0,
    0,
    0,
    0,
    0,
    0,
    55,
    64,
    0,
    0,
    0,
    0,
    0,
    0,
    56,
    64],
   'inline_axis_f64le': None,
   'xline_axis_f64le': None,
   'sample_axis_f32le': [0,
    0,
    0,
    0,
    0,
    0,
    128,
    64,
    0,
    0,
    0,
    65,
    0,
    0,
    64,
    65,
    0,
    0,
    128,
    65,
    0,
    0,
    160,
    65,
    0,
    0,
    192,
    65,
    0,
    0,
    224,
    65,
    0,
    0,
    0,
    66,
    0,
    0,
    16,
    66,
    0,
    0,
    32,
    66,
    0,
    0,
    48,
    66,
    0,
    0,
    64,
    66,
    0,
    0,
    80,
    66,
    0,
